In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime

# Set display options to see more data
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [18]:
# Path to my file
file_path = '/Users/abyudhka/Desktop/Project/australian-labor-market-analysis/Data/Raw/6291005.xlsx'

# display the sheets in the file
excel_file = pd.ExcelFile(file_path)
print("Available sheets")
for i, sheet in enumerate(excel_file.sheet_names, 1):
    print(f" {i}. {sheet}")

Available sheets
 1. Index
 2. Data1
 3. Data2
 4. Data3
 5. Enquiries


In [19]:
# load data sheets with no headers
df = pd.read_excel(file_path, sheet_name=['Data1','Data2','Data3'], nrows = 20, header = None)

print("Raw structure of data:")
for sheet_name, sheet_data in df.items():
    print(f"Sheet: {sheet_name}")
    print(f"Shape: {sheet_data.shape}")
    print(f"Columns: {sheet_data.columns.tolist()}")
    print(sheet_data.head(15))

Raw structure of data:
Sheet: Data1
Shape: (20, 251)
Columns: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 20

In [33]:
## concatenate the three sheets together and seperate metadata
# remove metadata
df_raw = pd.read_excel(file_path, sheet_name=['Data1','Data2','Data3'], skiprows=range(1,10))

# drop comman column in sheet 2 and sheet 3
df_raw['Data2'] = df_raw['Data2'].drop(columns=[df_raw['Data2'].columns[0]])
df_raw['Data3'] = df_raw['Data3'].drop(columns=[df_raw['Data3'].columns[0]])

# concatenate after dropping comman column
df_combined = pd.concat([df_raw['Data1'],df_raw['Data2'],df_raw['Data3']], axis=1)

# name the date column
df_combined = df_combined.rename(columns = {df_combined.columns[0]:'date'})
# print concatenated dataframe
print(" Combined data")
print(df_combined.shape)
print(df_combined.head())

 Combined data
(165, 541)
        date  \
0 1984-11-01   
1 1985-02-01   
2 1985-05-01   
3 1985-08-01   
4 1985-11-01   

   Australia ;  Agriculture, Forestry and Fishing ;  Employed total ;  \
0                                         410.719753                    
1                                         404.842338                    
2                                         394.461506                    
3                                         413.675791                    
4                                         433.581180                    

   Australia ;  Agriculture, Forestry and Fishing ;  > Employed full-time ;  \
0                                         338.746656                          
1                                         327.217337                          
2                                         319.238199                          
3                                         334.235215                          
4                                         3

In [36]:
## parse the series descriptions 
# check header column contents
for i, col in enumerate(df_combined.columns):
    print(f"{i}: {col}") 

# define function for parseing
for col in df_combined.columns:
    if col == 'date':
        continue
def parse_series_description(desc):
    """ Parses an ABS series description string into (state, industry, employment_type)."""
    # split on semicolon and clean each part
    parts = [p.strip().lstrip('>').strip() for p in desc.split(';')]
    parts = [p for p in parts if p]

    if len(parts) == 3:
        state     = parts[0]
        industry  = parts[1]
        emp_type  = parts[2]
    elif len(parts) == 2:
        state     = parts[0]
        industry  = 'All Industries'
        emp_type  = parts[1]
    else:
        state      = 'UNKNOWN'
        industry   = 'UNKNOWN'
        emp_type   = 'UNKNOWN'

    return state, industry, emp_type

# apply created function
parsed = {}
for col in df_combined.columns:
    if pd.isna(col) or 'date' in str(col): # skip date column
        continue

    state, industry, emp_type = parse_series_description(str(col)) # unpacking function to three seperate variables
    parsed[col] = {
        'state':    state,
        'industry': industry,
        'emp_type': emp_type
    }

# View the parsed result
parsed_df = pd.DataFrame(parsed).T
parsed_df.index.name = 'original_col_name'

print(parsed_df.head(10))
print()
print('Unique states:',     parsed_df['state'].nunique())
print('Unique industries:', parsed_df['industry'].nunique())
print('Unique emp_types:',  parsed_df['emp_type'].nunique())
print()
print(parsed_df['state'].unique())
print(parsed_df['industry'].unique())
print(parsed_df['emp_type'].unique())

0: date
1: Australia ;  Agriculture, Forestry and Fishing ;  Employed total ;
2: Australia ;  Agriculture, Forestry and Fishing ;  > Employed full-time ;
3: Australia ;  Agriculture, Forestry and Fishing ;  > Employed part-time ;
4: Australia ;  Mining ;  Employed total ;
5: Australia ;  Mining ;  > Employed full-time ;
6: Australia ;  Mining ;  > Employed part-time ;
7: Australia ;  Manufacturing ;  Employed total ;
8: Australia ;  Manufacturing ;  > Employed full-time ;
9: Australia ;  Manufacturing ;  > Employed part-time ;
10: Australia ;  Electricity, Gas, Water and Waste Services ;  Employed total ;
11: Australia ;  Electricity, Gas, Water and Waste Services ;  > Employed full-time ;
12: Australia ;  Electricity, Gas, Water and Waste Services ;  > Employed part-time ;
13: Australia ;  Construction ;  Employed total ;
14: Australia ;  Construction ;  > Employed full-time ;
15: Australia ;  Construction ;  > Employed part-time ;
16: Australia ;  Wholesale Trade ;  Employed total ;


In [44]:
## filter only required columns
# define major cities
major_city_states = [
    'Australia',         
    'New South Wales',  
    'Victoria',          
    'Queensland',       
    'South Australia',   
    'Western Australia'  
]

# filter for only major cities and employed total type
filtered = parsed_df[
    (parsed_df['state'].isin(major_city_states)) &
    (parsed_df['emp_type']=='Employed total')
]

print('Unique states:',     filtered['state'].nunique())
print('Unique industries:', filtered['industry'].nunique())
print('Unique emp_types:',  filtered['emp_type'].nunique())
print(filtered['state'].unique())
print(filtered['industry'].unique())
print(filtered['emp_type'].unique())

print(filtered.head(5))


Unique states: 6
Unique industries: 20
Unique emp_types: 1
['Australia' 'New South Wales' 'Victoria' 'Queensland' 'South Australia'
 'Western Australia']
['Agriculture, Forestry and Fishing' 'Mining' 'Manufacturing'
 'Electricity, Gas, Water and Waste Services' 'Construction'
 'Wholesale Trade' 'Retail Trade' 'Accommodation and Food Services'
 'Transport, Postal and Warehousing'
 'Information Media and Telecommunications'
 'Financial and Insurance Services'
 'Rental, Hiring and Real Estate Services'
 'Professional, Scientific and Technical Services'
 'Administrative and Support Services' 'Public Administration and Safety'
 'Education and Training' 'Health Care and Social Assistance'
 'Arts and Recreation Services' 'Other Services' 'All Industries']
['Employed total']
                                                        state  \
original_col_name                                               
Australia ;  Agriculture, Forestry and Fishing ...  Australia   
Australia ;  Mining ;  Empl

In [79]:
# select only required columns
cols_to_keep = filtered.index.tolist() 

# melt datadrame from wide to long

df_long = df_combined[['date'] + cols_to_keep].melt(
    id_vars='date',
    var_name='original_col',
    value_name='employed_thousands'
)

print(df_long.shape)


(19800, 3)


In [80]:
df_long['state'] = df_long['original_col'].map(filtered['state'])
df_long['industry'] = df_long['original_col'].map(filtered['industry']) 

df_long = df_long.drop(columns='original_col')

print(df_long.head())

        date  employed_thousands      state                           industry
0 1984-11-01          410.719753  Australia  Agriculture, Forestry and Fishing
1 1985-02-01          404.842338  Australia  Agriculture, Forestry and Fishing
2 1985-05-01          394.461506  Australia  Agriculture, Forestry and Fishing
3 1985-08-01          413.675791  Australia  Agriculture, Forestry and Fishing
4 1985-11-01          433.581180  Australia  Agriculture, Forestry and Fishing


In [82]:
# convert data 
df_long['employed_persons'] = (df_long['employed_thousands'] * 1000).round(0).astype('Int64')
df_long = df_long.drop(columns=['employed_thousands'])

# rename industry column
df_long['industry'] = df_long['industry'].str.lower()
df_long['industry'] = df_long['industry'].str.replace(', ', '_', regex=False)
df_long['industry'] = df_long['industry'].str.replace(' ', '_', regex=False)

# Rename 'All Industries' to 'total'
df_long['industry'] = df_long['industry'].replace('all_industries', 'total')


# reorder columns
df_long = df_long[['date', 'state', 'industry', 'employed_persons']]

# sort values
df_long = df_long.sort_values(['state', 'date', 'industry']).reset_index(drop=True)

# filter for post covid
df_long = df_long[df_long['date']>='2019-01-01']

df_timeseries_final = df_long.copy()

print("Final sorted data")
print("Unique states:", df_timeseries_final['state'].nunique())
print("Unique industries:", df_timeseries_final['industry'].nunique())
print("Unique dates:", df_timeseries_final['date'].nunique())
print("NaN values:", df_timeseries_final['employed_persons'].isna().sum())
print(df_timeseries_final.head())

Final sorted data
Unique states: 6
Unique industries: 20
Unique dates: 28
NaN values: 0
           date      state                             industry  \
2740 2019-02-01  Australia      accommodation_and_food_services   
2741 2019-02-01  Australia  administrative_and_support_services   
2742 2019-02-01  Australia     agriculture_forestry_and_fishing   
2743 2019-02-01  Australia         arts_and_recreation_services   
2744 2019-02-01  Australia                         construction   

      employed_persons  
2740            891178  
2741            436313  
2742            348146  
2743            253969  
2744           1145833  


In [83]:
# export file
# Save to processed folder
output_path = '/Users/abyudhka/Desktop/Project/australian-labor-market-analysis/Data/processed/industry_employment_timeseries.csv'

df_timeseries_final.to_csv(output_path, index=False)
print(f" Saved to: {output_path}")
print(f"   Rows: {len(df_timeseries_final)}")
print(f"   Columns: {len(df_timeseries_final.columns)}")

# Verification: Read back
df_test = pd.read_csv(output_path)
print(f"\n Verification: Successfully read back {len(df_test)} rows")
print("\nFile preview:")
print(df_test.head())

 Saved to: /Users/abyudhka/Desktop/Project/australian-labor-market-analysis/Data/processed/industry_employment_timeseries.csv
   Rows: 3360
   Columns: 4

 Verification: Successfully read back 3360 rows

File preview:
         date      state                             industry  \
0  2019-02-01  Australia      accommodation_and_food_services   
1  2019-02-01  Australia  administrative_and_support_services   
2  2019-02-01  Australia     agriculture_forestry_and_fishing   
3  2019-02-01  Australia         arts_and_recreation_services   
4  2019-02-01  Australia                         construction   

   employed_persons  
0            891178  
1            436313  
2            348146  
3            253969  
4           1145833  
